In [ ]:
import torch
from logger import log_loss_to_csv
import torchvision
import os
from ldm.models.diffusion.ddpm import LatentDiffusion
from ldm.models.diffusion.ddim import DDIMSampler
# from dataset import SimpleHandsDataset
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm

# --- PARAMÈTRES ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
max_images = 2000
n_epochs = 20
batch_size = 16
lr = 1e-5
save_dir = "./results_diffusion"
os.makedirs(save_dir, exist_ok=True)

# --- CONFIGURATION DU MODÈLE ---
unet_config = {
    "target": "ldm.modules.diffusionmodules.openaimodel.UNetModel",
    "params": {
        "image_size": 64, 
        "in_channels": 4,
        "out_channels": 4,
        "model_channels": 128,
        "attention_resolutions": [4, 2, 1],
        "num_res_blocks": 2,
        "channel_mult": [1, 2, 4, 4],
        "num_heads": 8,

        # Active l'attention croisée (Cross-Attention) pour le texte
        "use_spatial_transformer": True,
        "transformer_depth": 1,
        "context_dim": 1280, 

    }
}

cond_stage_config = {
    "target": "ldm.modules.encoders.modules.BERTEmbedder",
    "params": {
        "n_embed": 1280,
        "n_layer": 32
    }
}

first_stage_config = {
    "target": "ldm.models.autoencoder.AutoencoderKL",
    "params": {
        "embed_dim": 4,
        "monitor": "val/rec_loss",
        "ckpt_path": "vae_hands_latest.pt", 
        "ddconfig": {
            "double_z": True, "z_channels": 4, "resolution": 256,
            "in_channels": 3, "out_ch": 3, "ch": 128,
            "ch_mult": [1, 2, 4, 4], "num_res_blocks": 2, "attn_resolutions": []
        },
        "lossconfig": {"target": "torch.nn.Identity"}
    }
}

# 1. INSTANCIATION DU MODÈLE
print("Initialisation du modèle Latent Diffusion...")
model = LatentDiffusion(
    unet_config=unet_config,
    first_stage_config=first_stage_config,
    cond_stage_config=cond_stage_config,
    conditioning_key="crossattn", # Utilisation de l'attention croisée pour le texte
    timesteps=1000,
    beta_schedule="linear",
    linear_start=0.00085,
    linear_end=0.0120,
).to(device)

# Correction pour s'assurer que les buffers sont sur le bon device
model.logvar = model.logvar.to(device)

# Initialisation du Sampler DDIM pour la génération
sampler = DDIMSampler(model)
sampler.make_schedule(ddim_num_steps=50, ddim_eta=0.0, verbose=False)

# 2. PRÉPARATION DES DONNÉES
print("Chargement du dataset...")
full_dataset = SimpleHandsDataset(root_dir="./data/hands" , txt_prompts=True)

if len(full_dataset) > max_images:
    indices = torch.randperm(len(full_dataset))[:max_images]
    full_dataset = torch.utils.data.Subset(full_dataset, indices)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size)

# 3. OPTIMISEUR
# NOUVEAU : On n'optimise que le UNet (model.model), on garde BERT et le VAE gelés !
optimizer = torch.optim.AdamW(model.model.parameters(), lr=lr)

# 4. BOUCLE D'ENTRAÎNEMENT
print(f"Début de l'entraînement pour {n_epochs} epochs...")
best_val = float("inf")
for epoch in range(n_epochs):
    # --- PHASE TRAIN ---
    model.train()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{n_epochs}")
    for batch in pbar:
        # NOUVEAU : On sépare l'image et le texte de la liste "batch"
        img, prompts = batch 
        img = img.to(device)
        
        with torch.no_grad():
            # Encodage latent (Espace VAE)
            latents = model.get_first_stage_encoding(model.encode_first_stage(img))
            # NOUVEAU : Encodage du texte (Espace BERT)
            c = model.get_learned_conditioning(prompts)
        
        # Diffusion : on prédit le bruit
        t = torch.randint(0, model.num_timesteps, (latents.shape[0],), device=device).long()
        
        # NOUVEAU : On remplace "None" par "c" pour conditionner le UNet avec le texte
        loss, _ = model.p_losses(latents, c, t) 
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
    
    # --- PHASE VALIDATION ---
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for v_batch in val_loader:
            # NOUVEAU : On sépare aussi dans la validation
            v_img, v_prompts = v_batch 
            v_img = v_img.to(device)
            
            # On encode en latent
            v_latents = model.get_first_stage_encoding(model.encode_first_stage(v_img))
            # NOUVEAU : Encodage du texte de validation
            v_c = model.get_learned_conditioning(v_prompts)
            
            v_t = torch.randint(0, model.num_timesteps, (v_latents.shape[0],), device=device).long()
            # NOUVEAU : Conditionnement ici aussi
            v_l, _ = model.p_losses(v_latents, v_c, v_t)
            val_loss += v_l.item()
            
    avg_val_loss = val_loss / len(val_loader)
    print(f" --- Val Loss: {avg_val_loss:.4f} ---")

    if best_val > avg_val_loss :
      best_val = avg_val_loss
      torch.save(model.state_dict(), "latent_diffusion_hands_best_val.pt")

    csv_file_path = os.path.join(save_dir, "training_val_losses_prompts.csv")
    log_loss_to_csv(csv_file_path, epoch + 1, loss.item(), avg_val_loss)

    # --- GÉNÉRATION DE TEST ---
    print(f"Génération d'images de test...")
    with torch.no_grad():
        shape = [4, 64, 64] 
        
        # NOUVEAU : Il faut fournir un prompt au modèle pour qu'il génère l'image
        # Tu peux changer cette phrase par un vrai prompt de ton dataset
        test_prompts = ["a close up photo of a hand"] * 4 
        c_test = model.get_learned_conditioning(test_prompts)
        
        # NOUVEAU : On passe le conditionnement (c_test) au sampler au lieu de None
        samples_latent, _ = sampler.sample(S=50, conditioning=c_test, batch_size=4, shape=shape, verbose=False)
        
        samples_pixel = model.decode_first_stage(samples_latent)
        samples_pixel = torch.clamp((samples_pixel + 1.0) / 2.0, min=0.0, max=1.0)
        
        save_path = os.path.join(save_dir, f"sample_epoch_{epoch+1}.png")
        torchvision.utils.save_image(samples_pixel, save_path, nrow=2)
        print(f"Image sauvegardée : {save_path}")

    # Sauvegarde du modèle
    torch.save(model.state_dict(), "latent_diffusion_hands_prompts.pt")


    # --- GÉNÉRATION DE TEST (TEXT-TO-IMAGE) ---
    print(f"Génération d'images de test avec prompts...")
    model.eval() # On s'assure d'être en mode évaluation
    with torch.no_grad():
        # 1. Définition des prompts de test
        # On choisit 4 descriptions différentes pour tester la compréhension du modèle
        test_prompts = [
            "25 female fair with accessories",
            "50 male dark without accessories",
            "18 female dark with accessories",
            "40 male pale without accessories"
        ]
        
        # 2. Encodage du texte (Espace BERT)
        # On transforme nos phrases en vecteurs mathématiques
        c_test = model.get_learned_conditioning(test_prompts)
        
        # 3. Paramètres de génération
        shape = [4, 64, 64] # [Canaux latents, Hauteur latente, Largeur latente]
        batch_size = len(test_prompts) # 4 images
        
        # 4. Échantillonnage (Sampling DDIM)
        # NOUVEAU : On passe notre texte encodé (c_test) au paramètre "conditioning"
        samples_latent, _ = sampler.sample(
            S=50, # Nombre d'étapes de débruitage
            conditioning=c_test, 
            batch_size=batch_size, 
            shape=shape, 
            verbose=False
        )
        
        # 5. Décodage Latent -> Pixel (via le VAE)
        samples_pixel = model.decode_first_stage(samples_latent)
        
        # 6. Normalisation et Sauvegarde
        # On ramène les valeurs des pixels entre 0 et 1 pour l'image finale
        samples_pixel = torch.clamp((samples_pixel + 1.0) / 2.0, min=0.0, max=1.0)
        save_path = os.path.join(save_dir, f"sample_epoch_{epoch+1}.png")
        torchvision.utils.save_image(samples_pixel, save_path, nrow=2)
        
        print(f"Image sauvegardée : {save_path} (Prompts testés : {test_prompts})")

print("Entraînement terminé !")